# Upload Fraud Labels to Langfuse

This notebook uploads labels from the `fraud_labels` table in `/data/data.db` (inside the E2B template) into a Langfuse dataset for ground-truth comparison.

The upload is idempotent: dataset item IDs are deterministic hashes of `input` + `expected_output`, so re-running updates existing items instead of duplicating them.

In [1]:
%pip install aieng-agents

Note: you may need to restart the kernel to use updated packages.


In [ ]:
import json
import os
from pathlib import Path

from aieng.agents.tools import CodeInterpreter
from dotenv import load_dotenv

load_dotenv(verbose=True)

# Use the same template used in the CIBC Code Interpreter notebook.
TEMPLATE_NAME = "lobsuu8phhb16r4r04yr"
TEMPLATE_LABELS_PATH = "/data/data.db"
LANGFUSE_DATASET_NAME = "cibc-train-fraud-labels-ground-truth"

# Upload first MAX_ITEMS rows, in chunks.
MAX_ITEMS = 2220248
BATCH_SIZE = 200

required_env = ["LANGFUSE_PUBLIC_KEY", "LANGFUSE_SECRET_KEY", "LANGFUSE_HOST"]
missing = [name for name in required_env if not os.getenv(name)]
if missing:
    raise EnvironmentError(f"Missing Langfuse env vars: {missing}")

shared_code_interpreter = CodeInterpreter(template_name=TEMPLATE_NAME)
print(f"Ready. Template: {TEMPLATE_NAME}")

Ready. Template: lobsuu8phhb16r4r04yr


In [3]:
import asyncio
import json

def _extract_error_name(raw_result):
    # dict payload from tool
    if isinstance(raw_result, dict):
        err = raw_result.get("error")
        if isinstance(err, dict):
            return err.get("name")
        if hasattr(err, "name") and getattr(err, "name"):
            return str(getattr(err, "name"))
        if err is not None and "ContextRestarting" in str(err):
            return "ContextRestarting"
        return None

    # string payload from tool
    if isinstance(raw_result, str):
        try:
            parsed = json.loads(raw_result)
            if isinstance(parsed, dict):
                return _extract_error_name(parsed)
        except Exception:
            pass
        if "ContextRestarting" in raw_result:
            return "ContextRestarting"
        return None

    # object payload with .error or printable diagnostics
    err = getattr(raw_result, "error", None)
    if isinstance(err, dict):
        return err.get("name")
    if hasattr(err, "name") and getattr(err, "name"):
        return str(getattr(err, "name"))
    if err is not None and "ContextRestarting" in str(err):
        return "ContextRestarting"
    if "ContextRestarting" in str(raw_result):
        return "ContextRestarting"
    return None

def _is_transport_error(exc):
    msg = f"{type(exc).__name__}: {exc}"
    markers = [
        "RemoteProtocolError",
        "incomplete chunked read",
        "peer closed connection",
        "ReadTimeout",
        "ConnectTimeout",
    ]
    return any(marker in msg for marker in markers)

async def run_code_with_retry(code, retries=3):
    global shared_code_interpreter
    attempts = 0
    while True:
        try:
            result = await shared_code_interpreter.run_code(code=code)
        except Exception as exc:
            if attempts < retries and _is_transport_error(exc):
                attempts += 1
                wait_seconds = min(2 * attempts, 8)
                print(
                    f"Transport error detected. Reinitializing sandbox (attempt {attempts}/{retries})..."
                )
                await asyncio.sleep(wait_seconds)
                shared_code_interpreter = CodeInterpreter(template_name=TEMPLATE_NAME)
                _ = await shared_code_interpreter.run_code(code="print('sandbox_ready')")
                continue
            raise

        error_name = _extract_error_name(result)
        if error_name == "ContextRestarting" and attempts < retries:
            attempts += 1
            wait_seconds = min(2 * attempts, 8)
            print(f"Context restarting detected. Reinitializing sandbox (attempt {attempts}/{retries})...")
            await asyncio.sleep(wait_seconds)
            shared_code_interpreter = CodeInterpreter(template_name=TEMPLATE_NAME)
            # Warm up the new context to reduce immediate restart races.
            _ = await shared_code_interpreter.run_code(code="print('sandbox_ready')")
            continue
        return result

In [4]:
inspect_template_code = """
import json
import sqlite3
from pathlib import Path

db_path = Path(\"__TEMPLATE_DB_PATH__\")
if not db_path.exists():
    raise FileNotFoundError(f"Could not find database file: {db_path}")

with sqlite3.connect(db_path) as conn:
    cols = [row[1] for row in conn.execute("PRAGMA table_info(fraud_labels)").fetchall()]
    if not cols:
        raise RuntimeError("Table 'fraud_labels' was not found or has no columns.")
    total_rows = conn.execute("SELECT COUNT(*) FROM fraud_labels").fetchone()[0]

summary = {
    "db_path": str(db_path),
    "size_bytes": db_path.stat().st_size,
    "table": "fraud_labels",
    "columns": cols,
    "row_count": total_rows,
}

print(json.dumps(summary, indent=2))
"""

inspect_template_code = inspect_template_code.replace("__TEMPLATE_DB_PATH__", TEMPLATE_LABELS_PATH)
inspect_result = await run_code_with_retry(code=inspect_template_code, retries=3)
print(inspect_result)

{"stdout":["{","  \"db_path\": \"/data/data.db\",","  \"size_bytes\": 292700160,","  \"table\": \"fraud_labels\",","  \"columns\": [","    \"transaction_id\",","    \"fraud\"","  ],","  \"row_count\": 2220248","}"],"stderr":[],"results":null,"error":null}


In [2]:
import os

langfuse_public_key = os.getenv("LANGFUSE_PUBLIC_KEY")
langfuse_secret_key = os.getenv("LANGFUSE_SECRET_KEY")
langfuse_host = os.getenv("LANGFUSE_HOST")

In [9]:
# langfuse_public_key = os.getenv("LANGFUSE_PUBLIC_KEY")
# langfuse_secret_key = os.getenv("LANGFUSE_SECRET_KEY")
# langfuse_host = os.getenv("LANGFUSE_HOST")

# base_upload_config = {
#     "db_path": TEMPLATE_LABELS_PATH,
#     "dataset_name": LANGFUSE_DATASET_NAME,
#     "public_key": langfuse_public_key,
#     "secret_key": langfuse_secret_key,
#     "host": langfuse_host,
#     "max_items": MAX_ITEMS,
#     "batch_size": BATCH_SIZE,
# }

# upload_code_template = """
# import hashlib
# import json
# import sqlite3
# import subprocess
# import sys
# from pathlib import Path

# try:
#     from langfuse import Langfuse
# except ModuleNotFoundError:
#     subprocess.check_call([sys.executable, "-m", "pip", "install", "langfuse"])
#     from langfuse import Langfuse

# config = json.loads('''__UPLOAD_CONFIG_JSON__''')
# db_path = Path(config["db_path"])
# dataset_name = config["dataset_name"]
# offset = int(config["offset"])
# limit = int(config["limit"])
# is_first_batch = bool(config["is_first_batch"])
# if not db_path.exists():
#     raise FileNotFoundError(f"Could not find database file: {db_path}")

# def coerce_bool_like(value):
#     if isinstance(value, bool):
#         return value
#     if isinstance(value, (int, float)):
#         return bool(value)
#     if isinstance(value, str):
#         return value.strip().lower() in {"1", "true", "yes", "y", "fraud", "fraudulent"}
#     return bool(value)

# with sqlite3.connect(db_path) as conn:
#     cols = [row[1] for row in conn.execute("PRAGMA table_info(fraud_labels)").fetchall()]
#     if not cols:
#         raise RuntimeError("Table 'fraud_labels' was not found or has no columns.")

#     tx_col_candidates = ["transaction_id", "tx_id", "id"]
#     label_col_candidates = ["is_fraud", "fraud", "label", "is_laundering"]

#     tx_col = next((c for c in tx_col_candidates if c in cols), None)
#     label_col = next((c for c in label_col_candidates if c in cols), None)

#     if tx_col is None or label_col is None:
#         raise RuntimeError(
#             f"Could not infer required columns from fraud_labels. Found columns: {cols}"
#         )

#     query = f"SELECT {tx_col} AS transaction_id, {label_col} AS label_value FROM fraud_labels ORDER BY rowid LIMIT ? OFFSET ?"
#     rows = conn.execute(query, (limit, offset)).fetchall()

# client = Langfuse(
#     public_key=config["public_key"],
#     secret_key=config["secret_key"],
#     host=config["host"],
# )

# if is_first_batch:
#     try:
#         client.create_dataset(name=dataset_name)
#     except Exception:
#         pass

# uploaded = 0
# skipped = 0

# for rec_idx, (transaction_id, label_value) in enumerate(rows, start=offset + 1):
#     if transaction_id is None:
#         skipped += 1
#         continue

#     normalized = {
#         "input": {"transaction_id": str(transaction_id)},
#         "expected_output": {"is_fraud": coerce_bool_like(label_value)},
#         "metadata": {
#             "source": "data.db:fraud_labels",
#             "row_index": rec_idx,
#         },
#     }

#     canonical = json.dumps(
#         {"input": normalized["input"], "expected_output": normalized["expected_output"]},
#         sort_keys=True,
#         separators=(",", ":"),
#         ensure_ascii=True,
#     )
#     item_id = f"{dataset_name}:{hashlib.sha256(canonical.encode('utf-8')).hexdigest()}"

#     metadata = dict(normalized["metadata"])
#     metadata["id"] = str(rec_idx)

#     client.create_dataset_item(
#         dataset_name=dataset_name,
#         id=item_id,
#         input=normalized["input"],
#         expected_output=normalized["expected_output"],
#         metadata=metadata,
#     )
#     uploaded += 1

# client.flush()
# print(
#     json.dumps(
#         {
#             "offset": offset,
#             "limit": limit,
#             "uploaded_items": uploaded,
#             "skipped_items": skipped,
#         }
#     )
# )
# """

# total_uploaded = 0
# total_skipped = 0

# for offset in range(0, MAX_ITEMS, BATCH_SIZE):
#     limit = min(BATCH_SIZE, MAX_ITEMS - offset)
#     chunk_config = dict(base_upload_config)
#     chunk_config.update(
#         {
#             "offset": offset,
#             "limit": limit,
#             "is_first_batch": offset == 0,
#         }
#     )

#     upload_code = upload_code_template.replace(
#         "__UPLOAD_CONFIG_JSON__",
#         json.dumps(chunk_config),
#     )
#     chunk_result = await run_code_with_retry(code=upload_code, retries=3)
#     print(chunk_result)

#     parsed = None
#     if isinstance(chunk_result, str):
#         try:
#             parsed = json.loads(chunk_result)
#         except Exception:
#             parsed = None
#     elif isinstance(chunk_result, dict):
#         parsed = chunk_result

#     if isinstance(parsed, dict):
#         total_uploaded += int(parsed.get("uploaded_items", 0))
#         total_skipped += int(parsed.get("skipped_items", 0))

# print(
#     json.dumps(
#         {
#             "dataset_name": LANGFUSE_DATASET_NAME,
#             "max_items": MAX_ITEMS,
#             "batch_size": BATCH_SIZE,
#             "total_uploaded": total_uploaded,
#             "total_skipped": total_skipped,
#             "chunks": len(list(range(0, MAX_ITEMS, BATCH_SIZE))),
#         },
#         indent=2,
#     )
# )

In [10]:
count_code = """
import sqlite3
from pathlib import Path

db_path = Path('/data/data.db')
if not db_path.exists():
    raise FileNotFoundError(f'Could not find database file: {db_path}')

with sqlite3.connect(db_path) as conn:
    total = conn.execute('SELECT COUNT(*) FROM fraud_labels').fetchone()[0]

print(total)
"""

count_result = await run_code_with_retry(code=count_code, retries=3)
print(count_result)

{"stdout":["2220248"],"stderr":[],"results":null,"error":null}


In [4]:

# Fetch all items stored in the Langfuse dataset
from langfuse import Langfuse

LANGFUSE_DATASET_NAME = "cibc-train-fraud-labels-ground-truth"

lf = Langfuse(
    public_key=langfuse_public_key,
    secret_key=langfuse_secret_key,
    host=langfuse_host,
)

dataset = lf.get_dataset(LANGFUSE_DATASET_NAME)
dataset_items = [
    {"id": item.id, "input": item.input, "expected_output": item.expected_output}
    for item in dataset.items
]
dataset_ids = [item["id"] for item in dataset_items]

print(f"Total items in dataset '{LANGFUSE_DATASET_NAME}': {len(dataset_items)}")
print("\nFirst 3 items:")
for item in dataset_items[:3]:
    print(f"  id: {item['id']}")
    print(f"  input: {item['input']}")
    print(f"  expected_output: {item['expected_output']}")
    print()


Total items in dataset 'cibc-train-fraud-labels-ground-truth': 17923

First 3 items:
  id: cibc-train-fraud-labels-ground-truth:fraud:5813c1725fb5ff0c9a4cb4ed82acc2cde0ed6894b9bac8e959cd055fc0696374
  input: {'transaction_id': '18369831'}
  expected_output: {'is_fraud': True}

  id: cibc-train-fraud-labels-ground-truth:fraud:65ec9ce51de6c285e8ea0591ffc8d1cd041e5fbba603c671a9f5c37641eddc8a
  input: {'transaction_id': '8794556'}
  expected_output: {'is_fraud': True}

  id: cibc-train-fraud-labels-ground-truth:fraud:3f81da24fa72aea303491c2769719c5a4b68501b522977aca23e22dd4a1c5a1b
  input: {'transaction_id': '20650831'}
  expected_output: {'is_fraud': True}



In [21]:

import csv

# Extract transaction IDs and save to CSV
input_ids = [item["input"]["transaction_id"] for item in dataset_items]

csv_path = "dataset_input_ids.csv"
with open(csv_path, "w", newline="") as f:
    writer = csv.writer(f)
    writer.writerow(["transaction_id"])
    writer.writerows([[id_] for id_ in input_ids])

print(f"Exported {len(input_ids)} IDs to {csv_path}")


Exported 17923 IDs to dataset_input_ids.csv


In [20]:

# Count true/false in expected outcomes
expected_outputs = [item["expected_output"]["is_fraud"] for item in dataset_items]

true_count = sum(1 for v in expected_outputs if v is True)
false_count = sum(1 for v in expected_outputs if v is False)

print(f"is_fraud=True:  {true_count}")
print(f"is_fraud=False: {false_count}")


is_fraud=True:  2520
is_fraud=False: 15403


In [17]:

# Print all transaction IDs where is_fraud=True
fraud_ids = [item["input"]["transaction_id"] for item in dataset_items if item["expected_output"]["is_fraud"] is True]

print(f"Total fraud IDs (is_fraud=True): {len(fraud_ids)}")
for id_ in fraud_ids:
    print(id_)


Total fraud IDs (is_fraud=True): 1020
13153852
16150765
21333540
16505400
18082947
22268654
23695310
16473563
14121730
17314564
23364022
8049197
16402126
22611263
7965134
17259017
13916233
20857841
16336615
22934577
21717307
13680375
13974837
18632098
23066532
10746257
16812689
8164240
8858088
23198248
22932520
20772675
14238478
7750421
23353987
17998837
16766795
21698292
16520195
17189386
8728395
22857736
22703497
8311989
8047998
17325341
18720067
21333576
8774390
17543213
10996805
16159751
7936782
13133075
18278880
16422721
17590314
17669008
13910258
8278435
21838525
13181188
13463759
8338067
7806081
21333331
16635668
13369342
16850837
13181090
11491659
13830694
8813545
23023642
8351195
18540824
18006632
18007804
8378432
17842224
8450704
13587246
13145032
12368533
8349432
11096119
16533264
17113390
18631897
22876198
17506230
21830547
20936842
14201430
8062295
15951885
20580371
17931824
18377649
17604125
8367930
16733628
16766789
20651096
21243543
16609242
13368987
17517232
17243897
8

In [18]:

# Query fraud_labels in E2B for fraud=true rows and push 500 to Langfuse
# Schema: fraud_labels(transaction_id INTEGER, fraud TEXT)
import hashlib
import json

from langfuse import Langfuse

FRAUD_UPLOAD_COUNT = 2500

# --- Step 1: fetch fraud rows from E2B SQLite ---
fetch_fraud_code = """
import json
import sqlite3
from pathlib import Path

db_path = Path("/data/data.db")
if not db_path.exists():
    raise FileNotFoundError(f"Could not find database file: {db_path}")

with sqlite3.connect(db_path) as conn:
    distinct_vals = [row[0] for row in conn.execute("SELECT DISTINCT fraud FROM fraud_labels LIMIT 20").fetchall()]
    rows = conn.execute(
        "SELECT transaction_id FROM fraud_labels WHERE LOWER(TRIM(fraud)) IN ('true','1','yes','y','fraud','fraudulent') LIMIT 2500"
    ).fetchall()

print(json.dumps({"distinct": distinct_vals, "ids": [row[0] for row in rows]}))
"""

raw_result = await run_code_with_retry(code=fetch_fraud_code, retries=3)

def get_stdout_text(result):
    """Extract the printed stdout content from whatever run_code returns."""
    # If it's a string, check if it's a JSON wrapper like {"stdout": [...], ...}
    if isinstance(result, str):
        try:
            parsed = json.loads(result)
            if isinstance(parsed, dict) and ("stdout" in parsed or "output" in parsed or "text" in parsed):
                stdout = parsed.get("stdout") or parsed.get("output") or parsed.get("text") or ""
                if isinstance(stdout, list):
                    return "".join(stdout).strip()
                return str(stdout).strip()
        except Exception:
            pass
        return result.strip()
    if isinstance(result, dict):
        stdout = result.get("stdout") or result.get("output") or result.get("text") or ""
        if isinstance(stdout, list):
            return "".join(stdout).strip()
        return str(stdout).strip()
    return str(result).strip()

text = get_stdout_text(raw_result)
print(f"Raw result text: {repr(text[:300])}")

parsed_result = json.loads(text)

print(f"Distinct fraud values in DB: {parsed_result['distinct']}")
fraud_transaction_ids = parsed_result["ids"]
print(f"Fetched {len(fraud_transaction_ids)} fraud transaction IDs from E2B")

# --- Step 2: push to Langfuse ---
langfuse_public_key = os.getenv("LANGFUSE_PUBLIC_KEY")
langfuse_secret_key = os.getenv("LANGFUSE_SECRET_KEY")
langfuse_host = os.getenv("LANGFUSE_HOST")

lf = Langfuse(
    public_key=langfuse_public_key,
    secret_key=langfuse_secret_key,
    host=langfuse_host,
)

try:
    lf.create_dataset(name=LANGFUSE_DATASET_NAME)
except Exception:
    pass

uploaded = 0
for transaction_id in fraud_transaction_ids[:FRAUD_UPLOAD_COUNT]:
    input_ = {"transaction_id": str(transaction_id)}
    expected_output = {"is_fraud": True}

    canonical = json.dumps(
        {"input": input_, "expected_output": expected_output},
        sort_keys=True,
        separators=(",", ":"),
        ensure_ascii=True,
    )
    item_id = f"{LANGFUSE_DATASET_NAME}:fraud:{hashlib.sha256(canonical.encode('utf-8')).hexdigest()}"

    lf.create_dataset_item(
        dataset_name=LANGFUSE_DATASET_NAME,
        id=item_id,
        input=input_,
        expected_output=expected_output,
    )
    uploaded += 1

lf.flush()
print(f"Uploaded {uploaded} fraud transactions to dataset '{LANGFUSE_DATASET_NAME}'")


Raw result text: '{"distinct": ["No", "Yes"], "ids": [13803632, 13597072, 11416577, 7959875, 16308550, 16270866, 8049815, 7803991, 7849591, 21412126, 16500412, 7760583, 8299682, 8160719, 7804515, 21729902, 16954360, 18379895, 10751648, 16247457, 16813813, 18804141, 13250426, 16364405, 8827557, 17146448, 16472381, 836'
Distinct fraud values in DB: ['No', 'Yes']
Fetched 2500 fraud transaction IDs from E2B
Uploaded 2500 fraud transactions to dataset 'cibc-train-fraud-labels-ground-truth'
